# Лабораторная работа №1

## Компьютерная лексикография, терминологические базы (TBX) и Translation Memory (TMX) в Smartcat

**ФИО студента:** Климова Арина Дмитриевна
**Группа:** ИНФА-221
**Вариант:** 6 (предметная область: Финансы, банковское дело и платёжные системы)

---

**Цель:** научиться программно обрабатывать стандарты обмена терминологическими
данными (TBX) и памятью переводов (TMX), реализовать поиск нечётких совпадений
(Fuzzy Match) и подстановку терминов глоссария, экспортировать результат в
CAT-систему Smartcat.

**Порядок выполнения:**

1. Выполните ячейки Блока 1 (установка библиотек) и Блока 2 (данные).
2. Реализуйте четыре функции в Блоке 4 вместо маркеров `# TODO`.
3. Запустите Блок 5 — все `assert` должны пройти без ошибок.
4. Соберите файлы своего варианта, экспортируйте CSV, оформите отчёт.

**Оценивание (10 баллов):** парсинг TBX — 2, парсинг TMX — 2, Fuzzy Match — 3,
интеграция глоссария — 2, оформление — 1.

## Блок 1. Setup — установка библиотек

In [ ]:
!pip install -q Levenshtein pandas lxml

In [ ]:
from __future__ import annotations

import os
import re
import unicodedata
from typing import Any, Dict, List, Optional, Sequence, Tuple

import pandas as pd

try:
    from lxml import etree as ET

    LXML_AVAILABLE = True
except ImportError:
    import xml.etree.ElementTree as ET

    LXML_AVAILABLE = False

try:
    import Levenshtein as _lev

    def edit_distance(a: str, b: str) -> int:
        """Расстояние Левенштейна между двумя строками."""
        return _lev.distance(a, b)

except ImportError:

    def edit_distance(a: str, b: str) -> int:
        """Расстояние Левенштейна (резервная реализация на чистом Python)."""
        if a == b:
            return 0
        if not a:
            return len(b)
        if not b:
            return len(a)
        previous = list(range(len(b) + 1))
        for i, char_a in enumerate(a, start=1):
            current = [i]
            for j, char_b in enumerate(b, start=1):
                current.append(min(previous[j] + 1,
                                   current[j - 1] + 1,
                                   previous[j - 1] + (char_a != char_b)))
            previous = current
        return previous[-1]


# ВАЖНО: атрибут xml:lang в парсере раскрывается в полное имя с пространством имён
XML_LANG = "{http://www.w3.org/XML/1998/namespace}lang"

pd.set_option("display.max_colwidth", 60)
print("Окружение готово. XML backend:", "lxml" if LXML_AVAILABLE else "ElementTree")

Окружение готово. XML backend: lxml


## Блок 2. Теория: структура стандартов

### 2.1. TBX (TermBase eXchange, ISO 30042)

Формат обмена терминологическими базами. Организован **по понятиям**
(concept-oriented): одна `termEntry` = одно понятие, внутри — по одной `langSet`
на каждый язык.

```xml
<martif type="TBX" xml:lang="en">
  <martifHeader>…</martifHeader>
  <text><body>
    <termEntry id="tid-001">
      <descrip type="subjectField">Предметная область</descrip>
      <langSet xml:lang="en">
        <tig>
          <term>flight controller</term>
          <termNote type="partOfSpeech">noun</termNote>
        </tig>
      </langSet>
      <langSet xml:lang="ru">
        <ntig><termGrp>
          <term>полётный контроллер</term>
        </termGrp></ntig>
      </langSet>
    </termEntry>
  </body></text>
</martif>
```

| Тег | Назначение |
|-----|------------|
| `martif` | корень документа TBX-Core v2 |
| `termEntry` | терминологическая статья (одно понятие) |
| `langSet` | языковая секция, язык задаётся атрибутом `xml:lang` |
| `tig` / `ntig` | компактная / расширенная группа термина |
| `term` | сам термин |
| `termNote` | грамматическая или иная помета (`partOfSpeech`, `usage`) |
| `descrip` | описание: `definition`, `subjectField`, `context` |

### 2.2. TMX (Translation Memory eXchange, версия 1.4b)

Формат обмена памятью переводов. Организован **по сегментам**.

```xml
<tmx version="1.4">
  <header srclang="en" segtype="sentence" datatype="plaintext"
          creationtool="…" adminlang="en" o-tmf="TMX"/>
  <body>
    <tu tuid="1">
      <prop type="domain">Aerospace</prop>
      <tuv xml:lang="en"><seg>The autopilot holds the altitude.</seg></tuv>
      <tuv xml:lang="ru"><seg>Автопилот выдерживает высоту.</seg></tuv>
    </tu>
  </body>
</tmx>
```

| Тег | Назначение |
|-----|------------|
| `header` | метаданные памяти: инструмент, язык оригинала, тип сегментации |
| `tu` | translation unit — единица перевода |
| `tuv` | translation unit variant — вариант на одном языке (`xml:lang`) |
| `seg` | текст сегмента |
| `prop` | произвольное свойство (`domain`, `client`, `project`) |

### 2.3. Типы совпадений в CAT-системах

| Совпадение | Диапазон | Действие переводчика |
|------------|----------|----------------------|
| ICE (101 %) | 100 % + совпал контекст | подстановка без правки |
| Exact | 100 % | проверить контекст |
| High Fuzzy | 95–99 % | косметическая правка |
| Fuzzy | 75–94 % | редактирование |
| No Match | < 75 % | перевод с нуля |

## Блок 3. Данные

Ячейка ниже создаёт **демонстрационные** файлы `demo.tbx` и `demo.tmx` — они
нужны только для автотестов из Блока 5. Файлы своего варианта
(`variant_NN.tbx` — не менее 15 терминов, `variant_NN.tmx` — не менее 10
предложений) вы создаёте самостоятельно: либо аналогичным кодом, либо выгрузкой
из Smartcat, либо вручную.

In [ ]:
DEMO_TBX = """<?xml version="1.0" encoding="UTF-8"?>
<martif type="TBX" xml:lang="en">
  <martifHeader><fileDesc><titleStmt><title>Demo glossary</title></titleStmt>
  </fileDesc></martifHeader>
  <text><body>
    <termEntry id="tid-001">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en">
        <tig><term>flight controller</term>
        <termNote type="partOfSpeech">noun</termNote>
        <descrip type="definition">Onboard flight computer.</descrip></tig>
      </langSet>
      <langSet xml:lang="ru">
        <ntig><termGrp><term>полётный контроллер</term></termGrp></ntig>
      </langSet>
    </termEntry>
    <termEntry id="tid-002">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en"><tig><term>payload</term>
        <termNote type="partOfSpeech">noun</termNote></tig></langSet>
      <langSet xml:lang="ru"><ntig><termGrp><term>целевая нагрузка</term>
        </termGrp></ntig></langSet>
    </termEntry>
    <termEntry id="tid-003">
      <descrip type="subjectField">Demo</descrip>
      <langSet xml:lang="en"><tig><term>telemetry link</term></tig></langSet>
      <langSet xml:lang="ru"><ntig><termGrp><term>канал телеметрии</term>
        </termGrp></ntig></langSet>
    </termEntry>
  </body></text>
</martif>
"""

DEMO_TMX = """<?xml version="1.0" encoding="UTF-8"?>
<tmx version="1.4">
  <header creationtool="demo" creationtoolversion="1.0" segtype="sentence"
          o-tmf="TMX" adminlang="en" srclang="en" datatype="plaintext"/>
  <body>
    <tu tuid="1">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The flight controller stabilises the aircraft.</seg></tuv>
      <tuv xml:lang="ru"><seg>Полётный контроллер стабилизирует аппарат.</seg></tuv>
    </tu>
    <tu tuid="2">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The operator checks the telemetry link.</seg></tuv>
      <tuv xml:lang="ru"><seg>Оператор проверяет канал телеметрии.</seg></tuv>
    </tu>
    <tu tuid="3">
      <prop type="domain">Demo</prop>
      <tuv xml:lang="en"><seg>The payload is mounted under the fuselage.</seg></tuv>
      <tuv xml:lang="ru"><seg>Целевая нагрузка устанавливается под фюзеляжем.</seg></tuv>
    </tu>
  </body>
</tmx>
"""

with open("demo.tbx", "w", encoding="utf-8") as handler:
    handler.write(DEMO_TBX)
with open("demo.tmx", "w", encoding="utf-8") as handler:
    handler.write(DEMO_TMX)

print("Созданы файлы:", [f for f in os.listdir(".") if f.endswith((".tbx", ".tmx"))])

Созданы файлы: ['demo.tbx', 'demo.tmx', 'variant_6.tbx', 'variant_6.tmx']


## Блок 4. Задание — реализуйте четыре функции

Требования ко всем функциям: типизация из `typing`, соответствие PEP 8,
обработка отсутствующих элементов без падения (`None`-проверки).

**Подсказки:**

* язык секции читается как `element.get(XML_LANG)`, а не `element.get("lang")`;
* обход `<term>` внутри `langSet` удобно делать через `lang_set.iter("term")` —
  это покрывает и `tig`, и `ntig/termGrp`;
* текст сегмента лучше собирать через `"".join(seg.itertext())` — так не
  теряется текст при наличии инлайновых тегов;
* при подстановке терминов сортируйте их по убыванию длины.

In [ ]:
def parse_tbx(file_path: str, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """Разобрать TBX-глоссарий в плоскую таблицу терминологических пар."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    root = ET.parse(file_path).getroot()
    rows: List[Dict[str, str]] = []

    for entry in root.iter("termEntry"):
        subject = ""
        subject_node = entry.find("descrip[@type='subjectField']")
        if subject_node is not None and subject_node.text:
            subject = subject_node.text.strip()

        by_lang: Dict[str, Dict[str, str]] = {}

        for lang_set in entry.findall("langSet"):
            lang = (lang_set.get(XML_LANG) or "").split("-")[0].lower()
            terms = [t.text.strip() for t in lang_set.iter("term") if (t.text or "").strip()]
            if not terms:
                continue

            note = lang_set.find(".//termNote[@type='partOfSpeech']")
            definition = lang_set.find(".//descrip[@type='definition']")

            by_lang[lang] = {
                "term": terms[0],
                "pos": (note.text or "").strip() if note is not None else "",
                "definition": (definition.text or "").strip() if definition is not None else "",
            }

        if src_lang in by_lang and tgt_lang in by_lang:
            rows.append({
                "entry_id": entry.get("id", ""),
                "source_term": by_lang[src_lang]["term"],
                "target_term": by_lang[tgt_lang]["term"],
                "pos": by_lang[src_lang]["pos"] or by_lang[tgt_lang]["pos"],
                "definition": by_lang[src_lang]["definition"],
                "subject_field": subject,
            })

    return pd.DataFrame(
        rows,
        columns=["entry_id", "source_term", "target_term", "pos", "definition", "subject_field"],
    )


def parse_tmx(file_path: str, src_lang: str = "en", tgt_lang: str = "ru") -> pd.DataFrame:
    """Разобрать TMX-память переводов в таблицу параллельных сегментов."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(file_path)

    def _seg_text(tuv) -> str:
        seg = tuv.find("seg")
        if seg is None:
            return ""
        return re.sub(r"\s+", " ", "".join(seg.itertext())).strip()

    root = ET.parse(file_path).getroot()
    rows: List[Dict[str, Any]] = []

    for order, tu in enumerate(root.iter("tu"), start=1):
        segments: Dict[str, str] = {}

        for tuv in tu.findall("tuv"):
            lang = (tuv.get(XML_LANG) or tuv.get("lang") or "").split("-")[0].lower()
            text = _seg_text(tuv)
            if lang and text:
                segments[lang] = text

        if src_lang not in segments or tgt_lang not in segments:
            continue

        domain_node = tu.find("prop[@type='domain']")
        rows.append({
            "tuid": tu.get("tuid", str(order)),
            "source_segment": segments[src_lang],
            "target_segment": segments[tgt_lang],
            "domain": (domain_node.text or "").strip() if domain_node is not None else "",
            "src_len": len(segments[src_lang]),
        })

    frame = pd.DataFrame(
        rows,
        columns=["tuid", "source_segment", "target_segment", "domain", "src_len"],
    )
    frame["prev_source"] = frame["source_segment"].shift(1).fillna("")
    return frame


def calculate_fuzzy_match(source_query: str, tm_df: pd.DataFrame,
                          threshold: float = 70.0,
                          context_before: Optional[str] = None,
                          top_n: int = 5) -> List[Dict[str, Any]]:
    """Найти в памяти переводов сегменты, похожие на запрос."""
    if tm_df.empty:
        return []

    results: List[Dict[str, Any]] = []

    for _, row in tm_df.iterrows():
        norm_query = normalize(source_query)
        norm_segment = normalize(str(row["source_segment"]))

        lev = levenshtein_ratio(norm_query, norm_segment)
        tsr = token_sort_ratio(source_query, str(row["source_segment"]))
        score = max(lev, tsr)

        if score < threshold:
            continue

        context_matched = bool(
            context_before is not None
            and normalize(str(context_before)) == normalize(str(row.get("prev_source", "")))
        )

        results.append({
            "tuid": row["tuid"],
            "source_segment": row["source_segment"],
            "target_segment": row["target_segment"],
            "score": score,
            "lev_ratio": lev,
            "token_sort_ratio": tsr,
            "match_type": classify_match(score, context_matched),
        })

    results.sort(key=lambda item: item["score"], reverse=True)
    return results[:top_n]


def apply_term_glossary(text: str, glossary_df: pd.DataFrame,
                        template: str = "{source} [{target}]") -> str:
    """Разметить исходный текст переводными эквивалентами из глоссария."""
    if glossary_df.empty or not text:
        return text

    pairs = sorted(
        zip(glossary_df["source_term"], glossary_df["target_term"]),
        key=lambda pair: len(str(pair[0])),
        reverse=True,
    )

    protected: List[str] = []

    def _store(replacement: str) -> str:
        protected.append(replacement)
        return f"\x00{len(protected) - 1}\x00"

    result = text

    for source_term, target_term in pairs:
        if not source_term:
            continue

        pattern = re.compile(
            rf"(?<!\w){re.escape(str(source_term))}(?!\w)",
            re.IGNORECASE,
        )
        result = pattern.sub(
            lambda match: _store(
                template.format(source=match.group(0), target=target_term)
            ),
            result,
        )

    return re.sub(r"\x00(\d+)\x00", lambda m: protected[int(m.group(1))], result)

### 4.1. Вспомогательные функции (реализуйте при необходимости)

Эти функции не оцениваются отдельно, но нужны для `calculate_fuzzy_match`.

In [ ]:
def normalize(text: str) -> str:
    """Нормализовать строку: NFKC, нижний регистр, без пунктуации, одиночные пробелы."""
    text = unicodedata.normalize("NFKC", text).lower().replace("ё", "е")
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    return re.sub(r"\s+", " ", text).strip()


def levenshtein_ratio(a: str, b: str) -> float:
    """Процент сходства строк на основе расстояния Левенштейна."""
    if not a and not b:
        return 100.0
    longest = max(len(a), len(b))
    return round((1.0 - edit_distance(a, b) / longest) * 100.0, 2)


def token_sort_ratio(a: str, b: str) -> float:
    """Процент сходства при игнорировании порядка слов."""
    sorted_a = " ".join(sorted(normalize(a).split()))
    sorted_b = " ".join(sorted(normalize(b).split()))
    return levenshtein_ratio(sorted_a, sorted_b)


def classify_match(score: float, context_matched: bool) -> str:
    """Определить тип совпадения по проценту и признаку совпадения контекста."""
    if score >= 100.0:
        return "ICE (101%)" if context_matched else "Exact (100%)"
    if score >= 95.0:
        return "High Fuzzy"
    if score >= 75.0:
        return "Fuzzy"
    return "Low Fuzzy"

## Блок 5. Автотесты

Запустите ячейки ниже **после** реализации функций. Все проверки должны
завершиться без `AssertionError`. Тесты работают на демо-файлах из Блока 3.

In [ ]:
# --- Тест 1: метрики сходства -------------------------------------------
assert abs(levenshtein_ratio("telemetry", "telemetry") - 100.0) < 1e-6, \
    "Одинаковые строки должны давать 100 %"
assert levenshtein_ratio("", "") == 100.0, "Две пустые строки — 100 %"
assert 0.0 <= levenshtein_ratio("payload", "airframe") < 50.0, \
    "Непохожие строки должны давать низкий процент"
assert abs(token_sort_ratio("angle of attack", "attack of angle") - 100.0) < 1e-6, \
    "Token Sort Ratio должен игнорировать порядок слов"
assert normalize("  Angle   OF ATTACK, critical! ") == "angle of attack critical", \
    "normalize: нижний регистр, без пунктуации, одиночные пробелы"
print("Тест 1 (метрики сходства) — OK")

Тест 1 (метрики сходства) — OK


In [ ]:
# --- Тест 2: парсинг TBX -------------------------------------------------
glossary_df = parse_tbx("demo.tbx")

assert isinstance(glossary_df, pd.DataFrame), "parse_tbx должен вернуть DataFrame"
assert len(glossary_df) == 3, f"Ожидается 3 термина, получено {len(glossary_df)}"
for column in ["entry_id", "source_term", "target_term", "pos",
               "definition", "subject_field"]:
    assert column in glossary_df.columns, f"Нет колонки '{column}'"

row = glossary_df.loc[glossary_df["source_term"] == "flight controller"].iloc[0]
assert row["target_term"] == "полётный контроллер", "Неверный русский эквивалент"
assert row["entry_id"] == "tid-001", "Не извлечён атрибут id у termEntry"
assert row["pos"] == "noun", "Не извлечён termNote[@type='partOfSpeech']"
assert "telemetry link" in set(glossary_df["source_term"]), \
    "Термин из <tig> без termNote должен быть извлечён"
print("Тест 2 (парсинг TBX) — OK")
glossary_df.head()

Тест 2 (парсинг TBX) — OK


,entry_id,source_term,target_term,pos,definition,subject_field
0,tid-001,flight controller,полётный контроллер,noun,Onboard flight computer.,Demo
1,tid-002,payload,целевая нагрузка,noun,,Demo
2,tid-003,telemetry link,канал телеметрии,,,Demo


In [ ]:
# --- Тест 3: парсинг TMX -------------------------------------------------
tm_df = parse_tmx("demo.tmx")

assert isinstance(tm_df, pd.DataFrame), "parse_tmx должен вернуть DataFrame"
assert len(tm_df) == 3, f"Ожидается 3 единицы перевода, получено {len(tm_df)}"
for column in ["tuid", "source_segment", "target_segment", "domain",
               "src_len", "prev_source"]:
    assert column in tm_df.columns, f"Нет колонки '{column}'"

first = tm_df.iloc[0]
assert first["source_segment"].startswith("The flight controller"), \
    "Не извлечён текст <seg> исходного языка"
assert first["target_segment"].startswith("Полётный контроллер"), \
    "Не извлечён текст <seg> языка перевода"
assert first["prev_source"] == "", "У первой строки prev_source должен быть пустым"
assert tm_df.iloc[1]["prev_source"] == first["source_segment"], \
    "prev_source должен содержать предыдущий исходный сегмент"
assert first["src_len"] == len(first["source_segment"]), "Неверная длина сегмента"
print("Тест 3 (парсинг TMX) — OK")
tm_df.head()

Тест 3 (парсинг TMX) — OK


,tuid,source_segment,target_segment,domain,src_len,prev_source
0,1,The flight controller stabilises the aircraft.,Полётный контроллер стабилизирует аппарат.,Demo,46,
1,2,The operator checks the telemetry link.,Оператор проверяет канал телеметрии.,Demo,39,The flight controller stabilises the aircraft.
2,3,The payload is mounted under the fuselage.,Целевая нагрузка устанавливается под фюзеляжем.,Demo,42,The operator checks the telemetry link.


In [ ]:
# --- Тест 4: Fuzzy Match и ICE ------------------------------------------
exact_query = "The operator checks the telemetry link."
matches = calculate_fuzzy_match(exact_query, tm_df, threshold=70.0)
assert matches, "Точное совпадение обязано быть найдено"
assert abs(matches[0]["score"] - 100.0) < 1e-6, "Точное совпадение — 100 %"
assert matches[0]["match_type"] == "Exact (100%)", \
    "Без контекста 100 % — это Exact, а не ICE"

ice = calculate_fuzzy_match(
    exact_query, tm_df, threshold=70.0,
    context_before="The flight controller stabilises the aircraft.")
assert ice[0]["match_type"] == "ICE (101%)", \
    "При совпадении предыдущего сегмента 100 % должно стать ICE"

fuzzy = calculate_fuzzy_match("The engineer checks the telemetry link.",
                              tm_df, threshold=70.0)
assert fuzzy, "Похожий сегмент должен попадать в выдачу"
assert 75.0 <= fuzzy[0]["score"] < 100.0, \
    f"Ожидается Fuzzy 75–99 %, получено {fuzzy[0]['score']}"
assert fuzzy[0]["match_type"] in {"Fuzzy", "High Fuzzy"}, "Неверный тип совпадения"

scores = [m["score"] for m in calculate_fuzzy_match("telemetry link check",
                                                    tm_df, threshold=0.0)]
assert scores == sorted(scores, reverse=True), \
    "Результаты должны быть отсортированы по убыванию сходства"
assert calculate_fuzzy_match("Совершенно посторонний текст про бухгалтерию.",
                             tm_df, threshold=75.0) == [], \
    "Ниже порога результатов быть не должно"
assert len(calculate_fuzzy_match("telemetry", tm_df, threshold=0.0, top_n=2)) <= 2, \
    "Параметр top_n не учитывается"
print("Тест 4 (Fuzzy Match и ICE) — OK")

Тест 4 (Fuzzy Match и ICE) — OK


In [ ]:
# --- Тест 5: подстановка терминов ---------------------------------------
marked = apply_term_glossary("The flight controller reads the payload status.",
                             glossary_df)
assert "[полётный контроллер]" in marked, "Термин из глоссария не подставлен"
assert "[целевая нагрузка]" in marked, "Подставлен не каждый найденный термин"
assert "flight controller" in marked, "Исходный термин должен сохраняться"

assert apply_term_glossary("No terms here at all.", glossary_df) == \
    "No terms here at all.", "Текст без терминов не должен меняться"

case_test = apply_term_glossary("Payload mass is 2 kg.", glossary_df)
assert "Payload [целевая нагрузка]" in case_test, \
    "Поиск должен быть регистронезависимым, а регистр текста — сохраняться"

assert "[целевая нагрузка]" not in apply_term_glossary("payloads are heavy",
                                                       glossary_df), \
    "Совпадение должно быть по границам слов, а не по подстроке"
print("Тест 5 (подстановка терминов) — OK")
print(marked)

Тест 5 (подстановка терминов) — OK
The flight controller [полётный контроллер] reads the payload [целевая нагрузка] status.


## Блок 6. Работа со своим вариантом

После прохождения всех тестов выполните задание на данных своего варианта.

In [ ]:
VARIANT = 6
SURNAME = "Klimova"

TBX_FILE = f"variant_{VARIANT}.tbx"
TMX_FILE = f"variant_{VARIANT}.tmx"

SUBJECT_FIELD = "Финансы, банковское дело и платёжные системы"

TERMS_V6: List[Dict[str, str]] = [
    {"en": "microfinance", "ru": "микрофинансирование", "pos": "noun",
     "def_en": "Provision of financial services such as loans, savings, and insurance to low-income individuals."},
    {"en": "microfinance institution", "ru": "микрофинансовая организация", "pos": "noun",
     "def_en": "An organization that provides financial services to low-income clients, often referred to as an MFI."},
    {"en": "microcredit", "ru": "микрокредит", "pos": "noun",
     "def_en": "A small loan given to a low-income individual to help them become self-employed or grow a small business."},
    {"en": "microloan", "ru": "микрозайм", "pos": "noun",
     "def_en": "A small loan, typically of short duration, provided to microentrepreneurs or low-income borrowers."},
    {"en": "microentrepreneur", "ru": "микропредприниматель", "pos": "noun",
     "def_en": "An individual who operates a very small business, often with fewer than five employees."},
    {"en": "micro-insurance", "ru": "микрострахование", "pos": "noun",
     "def_en": "Insurance products designed for low-income individuals, with small premiums and coverage amounts."},
    {"en": "active loan", "ru": "активный заем", "pos": "noun",
     "def_en": "A loan that is currently outstanding and on which repayments are being made."},
    {"en": "active borrower", "ru": "активный заемщик", "pos": "noun",
     "def_en": "A client who currently has an outstanding loan balance with a microfinance institution."},
    {"en": "outstanding loan portfolio", "ru": "активный портфель займов", "pos": "noun",
     "def_en": "The total principal amount of all loans currently outstanding and owed to a lender."},
    {"en": "loan portfolio turnover", "ru": "оборот кредитного портфеля", "pos": "noun",
     "def_en": "The rate at which loans in a portfolio are repaid and replaced by new loans over a given period."},
    {"en": "portfolio quality", "ru": "качество портфеля", "pos": "noun",
     "def_en": "A measure of the credit risk in a loan portfolio, often assessed by repayment rates and arrears levels."},
    {"en": "loan loss provision", "ru": "резерв на убытки по займам", "pos": "noun",
     "def_en": "An expense set aside to cover expected losses from loans that may not be repaid."},
    {"en": "bad debt", "ru": "безнадежный долг", "pos": "noun",
     "def_en": "A loan that is considered uncollectible and is written off as a loss."},
    {"en": "defaulted loan", "ru": "просроченный заем", "pos": "noun",
     "def_en": "A loan on which the borrower has failed to make scheduled payments for a specified period."},
    {"en": "borrower in default", "ru": "заемщик, просрочивший выплату", "pos": "noun",
     "def_en": "A borrower who has failed to meet the legal obligations of a loan agreement."},
    {"en": "repayment plan", "ru": "график погашения", "pos": "noun",
     "def_en": "A schedule specifying the amounts and dates for repaying a loan."},
    {"en": "repayment frequency", "ru": "частота выплат", "pos": "noun",
     "def_en": "How often loan installments are due, such as weekly, biweekly, or monthly."},
    {"en": "balloon loan", "ru": "заем с погашением всей основной суммы в конце срока", "pos": "noun",
     "def_en": "A loan that requires a large principal payment at maturity rather than amortizing over the loan term."},
    {"en": "revolving credit", "ru": "возобновляемый кредит", "pos": "noun",
     "def_en": "A credit line that can be drawn down, repaid, and drawn down again up to a preset limit."},
    {"en": "working-capital loan", "ru": "заем на пополнение оборотного капитала", "pos": "noun",
     "def_en": "Short-term financing used to fund daily operations and cover the gap between expenses and revenue."},
    {"en": "group-based lending", "ru": "групповое кредитование", "pos": "noun",
     "def_en": "A lending methodology where loans are made to individuals within a group that shares liability for repayment."},
    {"en": "joint-liability group", "ru": "группа солидарной ответственности", "pos": "noun",
     "def_en": "A group of borrowers who mutually guarantee each other's loans, creating peer pressure for repayment."},
    {"en": "self-help group", "ru": "группа взаимопомощи", "pos": "noun",
     "def_en": "A voluntary association of people who save and lend money to each other, often without external support."},
    {"en": "savings account", "ru": "сберегательный счет", "pos": "noun",
     "def_en": "A deposit account held at a financial institution that earns interest and is used to store funds."},
    {"en": "compulsory savings", "ru": "гарантийный депозит", "pos": "noun",
     "def_en": "Savings that borrowers are required to deposit as a condition for obtaining a loan, often held as collateral."},
    {"en": "deposit", "ru": "вклад", "pos": "noun",
     "def_en": "Money placed into a financial institution for safekeeping or to earn interest."},
    {"en": "depositor", "ru": "вкладчик", "pos": "noun",
     "def_en": "A person or entity that places money into a deposit account at a financial institution."},
    {"en": "interest rate", "ru": "процентная ставка", "pos": "noun",
     "def_en": "The percentage charged on a loan or earned on savings, typically expressed as an annual rate."},
    {"en": "effective interest rate", "ru": "эффективная процентная ставка", "pos": "noun",
     "def_en": "The true cost of borrowing, taking into account compounding and all fees over a year."},
    {"en": "flat method", "ru": "метод начисления процентов на первоначальную сумму", "pos": "noun",
     "def_en": "Interest calculation based on the original principal amount for the entire loan term."},
    {"en": "declining-balance method", "ru": "метод начисления процентов на остаток задолженности", "pos": "noun",
     "def_en": "Interest calculation based on the remaining principal balance, resulting in declining interest charges."},
    {"en": "interest-rate margin", "ru": "маржа процентной ставки", "pos": "noun",
     "def_en": "The difference between the interest rate charged on loans and the rate paid on borrowed funds."},
    {"en": "financial margin", "ru": "финансовая маржа", "pos": "noun",
     "def_en": "The difference between financial income and financial costs, indicating profitability from lending."},
    {"en": "net interest margin", "ru": "чистая процентная маржа", "pos": "noun",
     "def_en": "Net interest income divided by average earning assets, showing the profitability of lending activities."},
    {"en": "operating expense", "ru": "операционные расходы", "pos": "noun",
     "def_en": "Costs incurred in the normal course of business operations, excluding financing and investment costs."},
    {"en": "administrative expense", "ru": "административные расходы", "pos": "noun",
     "def_en": "Costs related to the general administration and management of an organization."},
    {"en": "personnel expenses", "ru": "расходы на персонал", "pos": "noun",
     "def_en": "Costs associated with employing staff, including wages, benefits, and training."},
    {"en": "operating self-sufficiency", "ru": "операционная самоокупаемость", "pos": "noun",
     "def_en": "The ability of an institution to cover its operating costs from its operating income."},
    {"en": "financial self-sufficiency", "ru": "финансовая самоокупаемость", "pos": "noun",
     "def_en": "The ability to cover all costs, including the cost of capital, from operating income without subsidies."},
    {"en": "financial viability", "ru": "финансовая устойчивость", "pos": "noun",
     "def_en": "The ability of an organization to generate enough income to meet its financial obligations."},
    {"en": "outreach indicator", "ru": "индикатор охвата", "pos": "noun",
     "def_en": "A measure of the extent to which an MFI serves its target population."},
    {"en": "depth of outreach", "ru": "глубина охвата", "pos": "noun",
     "def_en": "The level of poverty of the clients served by a microfinance institution."},
    {"en": "impact analysis", "ru": "анализ воздействия", "pos": "noun",
     "def_en": "Assessment of the changes in clients' lives that can be attributed to microfinance services."},
    {"en": "benchmarking", "ru": "бенчмаркинг", "pos": "noun",
     "def_en": "Comparing an organization's performance metrics with those of peers or industry standards."},
    {"en": "ratio analysis", "ru": "анализ коэффициентов", "pos": "noun",
     "def_en": "A financial analysis technique that uses ratios to evaluate liquidity, profitability, and solvency."},
    {"en": "break-even analysis", "ru": "анализ безубыточности", "pos": "noun",
     "def_en": "A calculation of the point at which total revenue equals total costs, resulting in neither profit nor loss."},
    {"en": "cash flow analysis", "ru": "анализ денежных потоков", "pos": "noun",
     "def_en": "Examination of the inflows and outflows of cash to assess liquidity and financial health."},
    {"en": "balance sheet", "ru": "балансовый отчет", "pos": "noun",
     "def_en": "A financial statement showing assets, liabilities, and equity at a specific point in time."},
    {"en": "income statement", "ru": "отчет о прибылях и убытках", "pos": "noun",
     "def_en": "A financial statement showing revenues, expenses, and profits over a period."},
    {"en": "cash flow statement", "ru": "отчет о движении денежных средств", "pos": "noun",
     "def_en": "A financial statement showing cash inflows and outflows from operating, investing, and financing activities."},
    {"en": "audited financial statements", "ru": "финансовые отчеты, прошедшие аудиторскую проверку", "pos": "noun",
     "def_en": "Financial statements that have been examined and verified by an independent auditor."},
    {"en": "external audit", "ru": "внешний аудит", "pos": "noun",
     "def_en": "An audit conducted by an independent party outside the organization."},
    {"en": "internal audit", "ru": "внутренний аудит", "pos": "noun",
     "def_en": "An audit performed by an organization's own staff to assess internal controls and compliance."},
    {"en": "audit report", "ru": "аудиторский отчет", "pos": "noun",
     "def_en": "A written document presenting the auditor's findings and opinion on financial statements."},
    {"en": "internal control", "ru": "внутренний контроль", "pos": "noun",
     "def_en": "Processes and procedures designed to safeguard assets and ensure accurate financial reporting."},
    {"en": "accrual accounting", "ru": "бухгалтерский учет методом начисления", "pos": "noun",
     "def_en": "An accounting method that records revenues and expenses when they are incurred, not when cash changes hands."},
    {"en": "cash accounting", "ru": "кассовый метод ведения бухгалтерского учета", "pos": "noun",
     "def_en": "An accounting method that records transactions only when cash is received or paid."},
    {"en": "double-entry bookkeeping", "ru": "бухгалтерский учет на основе двойной записи", "pos": "noun",
     "def_en": "A system where every transaction is recorded with equal debits and credits."},
    {"en": "general ledger", "ru": "главная книга", "pos": "noun",
     "def_en": "A complete record of all financial transactions of an organization, organized by account."},
    {"en": "trial balance", "ru": "пробный баланс", "pos": "noun",
     "def_en": "A list of all general ledger accounts and their balances, used to check that debits equal credits."},
    {"en": "current asset", "ru": "оборотные активы", "pos": "noun",
     "def_en": "Assets that are expected to be converted to cash or used within one year."},
    {"en": "fixed asset", "ru": "основные средства", "pos": "noun",
     "def_en": "Long-term tangible assets used in operations, such as buildings, equipment, and vehicles."},
    {"en": "retained earnings", "ru": "аккумулированная накопленная прибыль", "pos": "noun",
     "def_en": "Profits that are reinvested in the business rather than distributed to owners."},
    {"en": "working capital", "ru": "оборотный капитал", "pos": "noun",
     "def_en": "The difference between current assets and current liabilities, measuring short-term liquidity."},
    {"en": "liquidity", "ru": "ликвидность", "pos": "noun",
     "def_en": "The ability of an organization to meet its short-term financial obligations."},
    {"en": "solvency", "ru": "платежеспособность", "pos": "noun",
     "def_en": "The ability of an organization to meet its long-term financial obligations."},
    {"en": "insolvency", "ru": "банкротство", "pos": "noun",
     "def_en": "A financial state where an organization cannot meet its debt obligations as they come due."},
    {"en": "leverage", "ru": "левередж", "pos": "noun",
     "def_en": "The use of borrowed funds to increase the potential return on investment."},
    {"en": "equity multiplier", "ru": "мультипликатор собственного капитала", "pos": "noun",
     "def_en": "A measure of financial leverage calculated as total assets divided by equity."},
    {"en": "return on assets", "ru": "рентабельность активов", "pos": "noun",
     "def_en": "A profitability ratio measuring net income generated per unit of assets."},
    {"en": "net profit", "ru": "чистая прибыль", "pos": "noun",
     "def_en": "The amount of profit remaining after all expenses and taxes have been deducted."},
    {"en": "operating profit", "ru": "операционная прибыль", "pos": "noun",
     "def_en": "Profit from core business operations, excluding interest and taxes."},
    {"en": "gross margin", "ru": "валовая прибыль", "pos": "noun",
     "def_en": "Revenue minus the direct cost of goods sold, expressed as a percentage or amount."},
    {"en": "profit margin", "ru": "норма прибыли", "pos": "noun",
     "def_en": "The ratio of net income to revenue, indicating profitability."},
    {"en": "cost recovery", "ru": "возмещение затрат", "pos": "noun",
     "def_en": "The extent to which an organization covers its costs through revenue from its services."},
    {"en": "subsidy", "ru": "субсидия", "pos": "noun",
     "def_en": "Financial assistance given to an organization to reduce costs or support operations."},
    {"en": "business plan", "ru": "бизнес-план", "pos": "noun",
     "def_en": "A written document describing a business's objectives, strategies, and financial forecasts."},
    {"en": "stakeholder", "ru": "заинтересованные стороны", "pos": "noun",
     "def_en": "Any party with an interest in an organization, including clients, staff, donors, and regulators."},
    {"en": "governance", "ru": "управление", "pos": "noun",
     "def_en": "The systems and processes by which an organization is directed and controlled."},
    {"en": "branchless banking", "ru": "внеофисное банковское обслуживание", "pos": "noun",
     "def_en": "Provision of financial services without traditional bank branches, often using mobile or agent networks."},
    {"en": "mobile banking", "ru": "мобильный банкинг", "pos": "noun",
     "def_en": "Financial services accessed through a mobile device such as a smartphone or tablet."},
    {"en": "remittance", "ru": "денежный перевод", "pos": "noun",
     "def_en": "Money sent by migrant workers to their home country or by one party to another."},
    {"en": "mortgage", "ru": "ипотека", "pos": "noun",
     "def_en": "A loan secured by real estate, typically used to purchase property."},
    {"en": "housing microfinance", "ru": "жилищное микрофинансирование", "pos": "noun",
     "def_en": "Small loans and financial services designed to help low-income households improve their housing."},
    {"en": "home loan", "ru": "жилищный кредит", "pos": "noun",
     "def_en": "A loan used to purchase, build, or renovate a home."},
    {"en": "leasing", "ru": "лизинг", "pos": "noun",
     "def_en": "A contract where one party uses an asset owned by another in exchange for periodic payments."},
    {"en": "lessor", "ru": "лизингодатель", "pos": "noun",
     "def_en": "The owner of an asset who leases it to another party."},
    {"en": "lessee", "ru": "лизингополучатель", "pos": "noun",
     "def_en": "The party who uses an asset under a lease agreement."},
    {"en": "collateral", "ru": "залог", "pos": "noun",
     "def_en": "An asset pledged as security for a loan, which can be seized if the borrower defaults."},
    {"en": "collateral substitute", "ru": "заменитель залога", "pos": "noun",
     "def_en": "An alternative to traditional physical collateral, such as group guarantees or character references."},
    {"en": "personal guarantee", "ru": "личное поручительство", "pos": "noun",
     "def_en": "A promise by an individual to repay a loan if the borrower defaults."},
    {"en": "loan guarantee", "ru": "гарантия по кредиту", "pos": "noun",
     "def_en": "A promise to repay a loan if the borrower fails to do so."},
    {"en": "secured debt", "ru": "обеспеченный долг", "pos": "noun",
     "def_en": "A debt backed by collateral that the lender can seize in case of default."},
    {"en": "debt service", "ru": "обслуживание долга", "pos": "noun",
     "def_en": "The cash required to pay interest and principal on a debt over a given period."},
    {"en": "credit manager", "ru": "менеджер по кредитованию", "pos": "noun",
     "def_en": "A staff member responsible for evaluating loan applications and managing credit risk."},
    {"en": "loan officer", "ru": "кредитный специалист", "pos": "noun",
     "def_en": "A staff member who works directly with borrowers, assessing loans and collecting repayments."},
    {"en": "client screening", "ru": "отбор клиентов", "pos": "noun",
     "def_en": "The process of evaluating potential borrowers to assess creditworthiness and eligibility."},
    {"en": "moral hazard", "ru": "моральный риск", "pos": "noun",
     "def_en": "The risk that a borrower will take excessive risks because someone else bears the cost."},
    {"en": "adverse selection", "ru": "неблагоприятный отбор", "pos": "noun",
     "def_en": "A situation where higher-risk borrowers are more likely to seek loans, potentially harming lenders."},
    {"en": "credit risk", "ru": "кредитный риск", "pos": "noun",
     "def_en": "The risk of loss from a borrower failing to repay a loan."},
    {"en": "currency risk", "ru": "валютный риск", "pos": "noun",
     "def_en": "The risk of financial loss from changes in exchange rates between currencies."},
    {"en": "interest-rate risk", "ru": "процентный риск", "pos": "noun",
     "def_en": "The risk of loss from changes in interest rates affecting the value of assets and liabilities."},
    {"en": "operating risk", "ru": "операционный риск", "pos": "noun",
     "def_en": "The risk of loss from inadequate or failed internal processes, people, or systems."},
    {"en": "liquidity risk", "ru": "риск ликвидности", "pos": "noun",
     "def_en": "The risk that an organization cannot meet its short-term financial obligations."},
    {"en": "capital requirements", "ru": "требования к капиталу", "pos": "noun",
     "def_en": "Minimum levels of capital that financial institutions must maintain as required by regulators."},
    {"en": "legal reserves", "ru": "обязательный резерв", "pos": "noun",
     "def_en": "Funds that financial institutions are required by law to set aside as a safety buffer."},
    {"en": "write-off", "ru": "списание", "pos": "noun",
     "def_en": "The removal of an asset from the balance sheet when it is deemed uncollectible."},
    {"en": "aging schedule", "ru": "график старения", "pos": "noun",
     "def_en": "A report that categorizes loans by the length of time payments are overdue."},
    {"en": "delinquency", "ru": "просроченная задолженность", "pos": "noun",
     "def_en": "The state of being overdue on a loan payment."},
    {"en": "arrears", "ru": "задолженность", "pos": "noun",
     "def_en": "Unpaid amounts that are past due on a loan."},
    {"en": "penalty interest", "ru": "пеня", "pos": "noun",
     "def_en": "Additional interest charged on overdue loan payments as a penalty."},
    {"en": "collection", "ru": "инкассо", "pos": "noun",
     "def_en": "The process of recovering overdue loan payments from borrowers."},
    {"en": "recovery", "ru": "возврат", "pos": "noun",
     "def_en": "The collection of loan amounts that were previously considered uncollectible."},
    {"en": "rescheduling of a loan", "ru": "пересмотр графика погашения займа", "pos": "noun",
     "def_en": "The modification of a loan's repayment schedule, usually to help a borrower in financial difficulty."},
    {"en": "certificate of deposit", "ru": "депозитный сертификат", "pos": "noun",
     "def_en": "A time deposit with a fixed term and interest rate, issued by a bank."},
    {"en": "savings bank", "ru": "сберегательный банк", "pos": "noun",
     "def_en": "A financial institution that primarily accepts savings deposits and provides loans."},
    {"en": "public development bank", "ru": "государственный банк развития", "pos": "noun",
     "def_en": "A government-owned financial institution that provides funding for development projects."},
    {"en": "apex institution", "ru": "апексная организация", "pos": "noun",
     "def_en": "A wholesale organization that provides funding and services to a network of retail microfinance institutions."},
    {"en": "downscaling", "ru": "даунскойлинг", "pos": "noun",
     "def_en": "The process by which commercial banks enter the microfinance market, serving lower-income clients."},
    {"en": "upscaling", "ru": "апскейлинг", "pos": "noun",
     "def_en": "The transformation of microfinance institutions into full-service banks or larger financial institutions."},
    {"en": "licensed financial institution", "ru": "лицензированная финансовая организация", "pos": "noun",
     "def_en": "A financial organization that has obtained a license from regulatory authorities to operate."},
]

PARALLEL_V6: List[Tuple[str, str]] = [
    ("The microfinance institution provides small loans to low-income entrepreneurs.",
     "Микрофинансовая организация предоставляет небольшие займы предпринимателям с низким доходом."),
    ("Microcredit helps borrowers start or expand their small businesses.",
     "Микрокредит помогает заемщикам начать или расширить свой малый бизнес."),
    ("The active loan portfolio grew by 15% over the last quarter.",
     "Активный портфель займов вырос на 15% за последний квартал."),
    ("A repayment schedule is agreed upon before the loan is disbursed.",
     "График погашения согласовывается до выдачи займа."),
    ("The loan-loss provision covers expected defaults in the portfolio.",
     "Резерв на убытки по займам покрывает ожидаемые невозвраты в портфеле."),
    ("Financial self-sufficiency means covering all costs from operating income.",
     "Финансовая самоокупаемость означает покрытие всех расходов за счет операционного дохода."),
    ("Compulsory savings are often required as collateral for a loan.",
     "Гарантийный депозит часто требуется в качестве залога по займу."),
    ("Group-based lending uses joint liability to ensure repayment.",
     "Групповое кредитование использует солидарную ответственность для обеспечения возврата."),
    ("The operating expense ratio measures administrative efficiency.",
     "Коэффициент операционных расходов измеряет административную эффективность."),
    ("Interest rate margins must cover the cost of funds and risk.",
     "Маржа процентной ставки должна покрывать стоимость привлеченных средств и риск."),
    ("The outstanding loan portfolio includes all active loans.",
     "Активный портфель займов включает все действующие займы."),
    ("A borrower in default may face legal action and penalty interest.",
     "Заемщик, просрочивший выплаты, может столкнуться с судебным иском и пеней."),
    ("The balance sheet shows assets, liabilities, and equity.",
     "Балансовый отчет показывает активы, обязательства и собственный капитал."),
    ("Cash flow analysis helps assess the liquidity of the institution.",
     "Анализ денежных потоков помогает оценить ликвидность организации."),
    ("The audit report provides an opinion on the financial statements.",
     "Аудиторский отчет содержит заключение о финансовой отчетности."),
    ("Operational sustainability is achieved when revenues cover operating costs.",
     "Операционная самоокупаемость достигается, когда доходы покрывают операционные расходы."),
    ("The social performance indicator measures depth of outreach.",
     "Индикатор социальной эффективности измеряет глубину охвата."),
    ("Savings accounts allow clients to deposit and withdraw funds safely.",
     "Сберегательные счета позволяют клиентам безопасно вносить и снимать средства."),
    ("The loan officer assesses the creditworthiness of each applicant.",
     "Кредитный специалист оценивает кредитоспособность каждого заявителя."),
    ("Financial statements must be prepared in accordance with applicable standards.",
     "Финансовая отчётность должна составляться в соответствии с применимыми стандартами."),
    ("Business cycles refer to the periods of various success, struggle, and medium-quality profits encountered by companies in the normal course of the economy.",
     "Бизнес-циклы — это периоды различного успеха, трудностей и среднего уровня прибыли, с которыми компании сталкиваются в нормальном ходе экономики."),
    ("These periods affect every individual.",
     "Эти периоды затрагивают каждого человека."),
    ("The difference isn't the business, but rather, is the economy.",
     "Разница не в бизнесе, а в экономике."),
    ("When the economy is good, most businesses experience a boom, or an increase in profits and success.",
     "Когда экономика в хорошем состоянии, большинство компаний переживают бум, то есть рост прибыли и успеха."),
    ("When people have more money to spend, businesses have more money to make.",
     "Когда у людей больше денег для расходов, у компаний больше возможностей заработать."),
    ("Businesses experience a bust, or a decrease in profits and success, when the economy falters.",
     "Компании переживают спад, то есть снижение прибыли и успеха, когда экономика ослабевает."),
    ("For businesses, a sagging economy means it might be difficult to stay in operation.",
     "Для компаний падающая экономика означает, что может быть трудно продолжать деятельность."),
    ("Business contractions are normal periods of reduction in business after prolonged growth.",
     "Сокращения деловой активности — это нормальные периоды спада в бизнесе после продолжительного роста."),
    ("Business contractions occur regularly and vary in severity.",
     "Сокращения деловой активности происходят регулярно и различаются по степени тяжести."),
    ("Employers will require a smaller amount of help because consumers are purchasing less.",
     "Работодателям потребуется меньше рабочей силы, поскольку потребители покупают меньше."),
    ("Unemployment will accordingly increase, and wages will fall.",
     "Соответственно вырастет безработица, а заработные платы упадут."),
    ("Recessions are multi-month-long declines in wages, general economic activity, and most importantly, GDP.",
     "Рецессии — это продолжающиеся несколько месяцев снижения заработных плат, общей экономической активности и, что наиболее важно, ВВП."),
    ("Recessions are more serious than business contractions.",
     "Рецессии серьезнее, чем сокращения деловой активности."),
    ("Recessions last longer than business contractions, can be more severe, and can signal larger problems in the economy.",
     "Рецессии длятся дольше, чем сокращения деловой активности, могут быть более тяжелыми и сигнализировать о более крупных проблемах в экономике."),
    ("Inflation is the decrease in value relative to overall quantity and production.",
     "Инфляция — это снижение стоимости относительно общего количества и объёма производства."),
    ("Central banks are the institutions tasked with managing countries' economies.",
     "Центральные банки — это учреждения, на которые возложено управление экономикой стран."),
    ("Central banks have responded to population growth by minting more money.",
     "Центральные банки отреагировали на рост населения, начав выпускать больше денег."),
    ("Not having enough money in circulation could lead to panics.",
     "Недостаток денег в обращении может привести к паникам."),
    ("Panics are economic downturns that are usually accompanied by anxiety over currency.",
     "Паники — это экономические спады, которые обычно сопровождаются тревогой по поводу валюты."),
    ("Central banks release more dollar bills and coins regularly, based upon a pre-planned schedule.",
     "Центральные банки регулярно выпускают больше банкнот и монет по заранее спланированному графику."),
    ("As additional currency has come into circulation, its value has decreased.",
     "По мере поступления дополнительной валюты в обращение её стоимость снижалась."),
    ("The process of inflation means that money is worth less over time.",
     "Процесс инфляции означает, что деньги со временем обесцениваются."),
    ("Even though wages might increase in amount, they will have a lesser purchasing power.",
     "Даже если заработные платы могут расти в сумме, их покупательная способность будет меньше."),
    ("Purchasing power is a measure of how many goods and/or services currency can be exchanged for.",
     "Покупательная способность — это мера того, на какое количество товаров и/или услуг можно обменять валюту."),
    ("Deflation is the process of a currency becoming more valuable due to a tight production schedule.",
     "Дефляция — это процесс повышения стоимости валюты из-за ограниченного графика производства."),
    ("If there was less currency around today, each dollar would be worth more.",
     "Если бы сегодня в обращении было меньше валюты, каждый доллар стоил бы больше."),
    ("Recessions last longer than panics and could be indicative of large-scale economic downturns.",
     "Рецессии длятся дольше паник и могут свидетельствовать о масштабных экономических спадах."),
]

def build_tbx(terms: Sequence[Dict[str, str]], path: str,
              subject_field: str = SUBJECT_FIELD) -> str:
    martif = ET.Element("martif", {"type": "TBX", XML_LANG: "en"})

    header = ET.SubElement(martif, "martifHeader")
    file_desc = ET.SubElement(header, "fileDesc")
    title_stmt = ET.SubElement(file_desc, "titleStmt")
    ET.SubElement(title_stmt, "title").text = f"Glossary EN-RU: {subject_field}"
    source_desc = ET.SubElement(file_desc, "sourceDesc")
    ET.SubElement(source_desc, "p").text = "Lab 01, variant 6"

    body = ET.SubElement(ET.SubElement(martif, "text"), "body")

    for index, item in enumerate(terms, start=1):
        entry = ET.SubElement(body, "termEntry", {"id": f"tid-{index:03d}"})
        ET.SubElement(entry, "descrip", {"type": "subjectField"}).text = subject_field

        lang_en = ET.SubElement(entry, "langSet", {XML_LANG: "en"})
        tig = ET.SubElement(lang_en, "tig")
        ET.SubElement(tig, "term").text = item["en"]
        ET.SubElement(tig, "termNote", {"type": "partOfSpeech"}).text = item["pos"]
        ET.SubElement(tig, "descrip", {"type": "definition"}).text = item.get("def_en", "")

        lang_ru = ET.SubElement(entry, "langSet", {XML_LANG: "ru"})
        term_grp = ET.SubElement(ET.SubElement(lang_ru, "ntig"), "termGrp")
        ET.SubElement(term_grp, "term").text = item["ru"]
        ET.SubElement(term_grp, "termNote", {"type": "partOfSpeech"}).text = item["pos"]

    xml_body = ET.tostring(martif, encoding="unicode")
    with open(path, "w", encoding="utf-8") as handler:
        handler.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        handler.write(xml_body)
    return path


def build_tmx(pairs: Sequence[Tuple[str, str]], path: str,
              src_lang: str = "en", tgt_lang: str = "ru") -> str:
    tmx = ET.Element("tmx", {"version": "1.4"})
    ET.SubElement(tmx, "header", {
        "creationtool": "lab01-generator",
        "creationtoolversion": "1.0",
        "segtype": "sentence",
        "o-tmf": "TMX",
        "adminlang": "en",
        "srclang": src_lang,
        "datatype": "plaintext",
    })
    body = ET.SubElement(tmx, "body")

    for index, (source, target) in enumerate(pairs, start=1):
        tu = ET.SubElement(body, "tu", {"tuid": str(index)})
        ET.SubElement(tu, "prop", {"type": "domain"}).text = "Finance / Banking"
        for lang, text in ((src_lang, source), (tgt_lang, target)):
            tuv = ET.SubElement(tu, "tuv", {XML_LANG: lang})
            ET.SubElement(tuv, "seg").text = text

    xml_body = ET.tostring(tmx, encoding="unicode")
    with open(path, "w", encoding="utf-8") as handler:
        handler.write('<?xml version="1.0" encoding="UTF-8"?>\n')
        handler.write(xml_body)
    return path


build_tbx(TERMS_V6, TBX_FILE)
build_tmx(PARALLEL_V6, TMX_FILE)

my_glossary = parse_tbx(TBX_FILE)
my_tm = parse_tmx(TMX_FILE)

print(f"Глоссарий: {len(my_glossary)} терминов")
print(f"Память переводов: {len(my_tm)} сегментов")

QUERIES_V6 = [
    {
        "label": "1. ICE — точное совпадение с совпадением контекста",
        "query": "The active loan portfolio grew by 15% over the last quarter.",
        "context": "Microcredit helps borrowers start or expand their small businesses.",
    },
    {
        "label": "2. Fuzzy — заменено одно слово",
        "query": "The microfinance institution provides small loans to low-income borrowers.",
        "context": None,
    },
    {
        "label": "3. No match — сегмент вне предметной области",
        "query": "The flight controller stabilises the airframe during autonomous flight.",
        "context": None,
    },
]

all_matches: List[Dict[str, Any]] = []

for case in QUERIES_V6:
    matches = calculate_fuzzy_match(
        case["query"],
        my_tm,
        threshold=70.0,
        context_before=case["context"],
    )
    print("=" * 100)
    print(case["label"])
    print(f"Запрос: {case['query']}")

    if not matches:
        print("  -> совпадений выше порога 70 % не найдено")
        continue

    for rank, match in enumerate(matches, start=1):
        print(f"  #{rank} {match['score']:6.2f} %  [{match['match_type']:<12}] tuid={match['tuid']}")
        print(f"      TM src: {match['source_segment']}")
        print(f"      TM tgt: {match['target_segment']}")
        match["query"] = case["query"]
        all_matches.append(match)

matches_df = pd.DataFrame(
    all_matches,
    columns=[
        "query", "tuid", "source_segment", "target_segment",
        "score", "lev_ratio", "token_sort_ratio", "match_type",
    ],
)

smartcat_df = pd.DataFrame({
    "Source (en)": my_glossary["source_term"],
    "Target (ru)": my_glossary["target_term"],
    "Comment": my_glossary["definition"],
})
smartcat_df.to_csv("glossary_smartcat.csv", index=False, encoding="utf-8-sig")

print("=" * 100)
print("Созданы файлы:")
for name in [TBX_FILE, TMX_FILE, "glossary_smartcat.csv"]:
    if os.path.exists(name):
        print(f"  {name:<30} {os.path.getsize(name):>7} байт")

if not matches_df.empty:
    display(matches_df[["query", "tuid", "score", "match_type"]])

Глоссарий: 121 терминов
Память переводов: 47 сегментов
1. ICE — точное совпадение с совпадением контекста
Запрос: The active loan portfolio grew by 15% over the last quarter.
  #1 100.00 %  [ICE (101%)  ] tuid=3
      TM src: The active loan portfolio grew by 15% over the last quarter.
      TM tgt: Активный портфель займов вырос на 15% за последний квартал.
2. Fuzzy — заменено одно слово
Запрос: The microfinance institution provides small loans to low-income borrowers.
  #1  89.61 %  [Fuzzy       ] tuid=1
      TM src: The microfinance institution provides small loans to low-income entrepreneurs.
      TM tgt: Микрофинансовая организация предоставляет небольшие займы предпринимателям с низким доходом.
3. No match — сегмент вне предметной области
Запрос: The flight controller stabilises the airframe during autonomous flight.
  -> совпадений выше порога 70 % не найдено
Созданы файлы:
  variant_6.tbx                    67813 байт
  variant_6.tmx                    17120 байт
  glossary_s

,query,tuid,score,match_type
0,The active loan portfolio grew by 15% over the last quar...,3,100.00,ICE (101%)
1,The microfinance institution provides small loans to low...,1,89.61,Fuzzy


## Блок 7. Выводы

Ответьте письменно (3–5 предложений на пункт):

1. Чем концептуально-ориентированная организация TBX отличается от
   сегментно-ориентированной организации TMX?
   
   TBX организован вокруг понятия: одна termEntry описывает одно понятие, а внутри неё langSet хранит термины, часть речи и определение для каждого языка. TMX организован вокруг готового перевода: одна tu - это пара сегментов tuv/seg на исходном и целевом языках. В итоге из TBX получается таблица терминов, а из TMX — таблица параллельных сегментов с prev_source для контекста.

2. Почему `xml:lang` требует особой обработки при парсинге и как вы её решили?
   
   В XML-парсере атрибут xml:lang раскрывается в полное имя {http://www.w3.org/XML/1998/namespace} lang, поэтому вызов element.get("lang") вернул бы None, и языковые секции не были бы распознаны. Я решила это константой XML_LANG и чтением lang_set.get(XML_LANG), а также нормализацией кода языка через split("-")[0].lower(), чтобы корректно обрабатывать варианты вроде en-US.

3. В каких случаях Levenshtein ratio даёт заниженную оценку сходства и как это
   компенсирует Token Sort Ratio?

   Левенштейн чувствителен к порядку слов: при перестановке расстояние большое, хотя смысл тот же - в работе такая пара дала lev = 30, а token_sort = 100. Также она сильнее штрафует вставки, удаления и морфологические различия. Token Sort Ratio нормализует строки, сортирует токены и сравнивает уже отсортированные последовательности, поэтому игнорирует порядок слов. Итоговый балл берётся как максимум двух метрик, что повышает полноту выдачи, не теряя точность на почти полных совпадениях.

4. Чем ICE (101 %) отличается от обычного 100 %-совпадения и почему это важно
   для стоимости проекта в Smartcat?

   Exact - это полное совпадение сегмента без учёта окружения, его всё равно нужно проверить по контексту. ICE - это 100 % плюс совпадение предыдущего сегмента, поэтому перевод можно подставить без правки. В реализации context_before сравнивается с prev_source кандидата, и только при совпадении тип повышается до ICE (101%). Для стоимости проекта это важно: ICE обычно оплачивается по минимальной ставке или не оплачивается вовсе, а Fuzzy 75–99 % - частично, поэтому рост доли ICE снижает затраты на редактуру и проверку.

5. Какие проблемы возникли при автоматической подстановке терминов и как вы их
   решили?
   
   Вложенность: короткий термин может разрушить длинный (например "loan" внутри "loan officer"), поэтому термины сортируются по убыванию длины.
   Cовпадение по подстроке ("payloads" не должен давать «payload»), для решения выставлены границы слов (?<!\w)...(?!\w).
   Повторная разметка уже подставленных фрагментов, для неё используются временные маркеры \x00...\x00, которые затем заменяются на готовые подстановки.
**Ваши выводы:**
В результате выполнения работы я научилась программно обрабатывать стандарты обмена терминологическими данными (TBX) и память переводов (TMX), реализовывать поиск нечётких совпадений (Fuzzy Match) и подстановку терминов глоссария, экспортировать результат в CAT-систему Smartcat.

---

### Что сдать на проверку

1. Ссылка на Colab-ноутбук `lab_01_Фамилия.ipynb` (доступ «для всех по ссылке»).
2. Файлы `variant_NN.tbx`, `variant_NN.tmx`, `glossary_smartcat.csv`.
3. Отчёт с выводами и скриншотом импортированного глоссария в Smartcat.